In [2]:
%pip install pandas ipywidgets

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   --------- ------------------------------ 2.4/9.6 MB 17.3 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.6 MB 21.7 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 20.9 MB/s  0:00:00

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   ---------------

In [2]:
%pip install -U daschlab

  Using cached daschlab-1.0.0-py3-none-any.whl
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pywwt-0.26.0-py3-none-any.whl
  Using cached toasty-0.20.1-cp313-cp313-win_amd64.whl
  Using cached pyerfa-2.0.1.5-cp39-abi3-win_amd64.whl.metadata (5.9 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached ipyevents-2.0.4-py3-none-any.whl.metadata (3.0 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached reproject-0.21.0-cp311-abi3-win_amd64.whl.metadata (3.5 kB)
  Using cached wwt_data_formats-0.18.1-py3-none-any.whl
  Using cached astropy_healpix-2.0.1-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached dask_image-2026.5.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached pyavm-0.9.9-py3-none-any.whl.metadata (5.7 kB)
  Using cached part

In [ ]:
# NOTE : dasch_Cutout_Recreation.ipynb - Cell 1
# Batch cutout extraction that ONLY downloads the plates DASCH itself uses.
# For every star in a targets .txt file, a cutout is downloaded for each exposure that is
# detected in DASCH's lightcurve AND not flagged by lc.apply_standard_rejections()
# (i.e. Stage 4 survivors from the funnel).
# Output layout matches 01_Load_Cutout, so 02_A / 03_A / 03_B pick the folders up unchanged:
#   data/<Star>/cutouts/*.fits, data/<Star>/plate_manifest.csv,
#   data/<Star>/<Star>_Cutout_Download_Summary.txt, data/download_Summary_<date>_<time>.txt

# Re-running this cell while an earlier run is still going would start a second set of
# downloads on top of it (and freeze the first run's progress bar) -- so stop here instead.
_prev_thread = globals().get("_batch_thread")
if _prev_thread is not None and _prev_thread.is_alive():
    class _StopCell(Exception):
        def _render_traceback_(self):
            return [str(self)]
    raise _StopCell("An earlier run of this cell is still working. Press Pause on that run "
                    "(or wait for it to finish), then re-run this cell.")

from IPython.display import HTML, display
from daschlab import open_session
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from importlib import metadata
import html
import platform
import re
import shutil
import sys
import threading
import time
import traceback
import numpy as np
import pandas as pd
import ipywidgets as widgets

display(HTML("""
<style>
.output, .output_text, .output_stream, .output_stdout {
    color: white !important;
}
.dark-log-output,
.dark-log-output .jp-OutputArea-output,
.dark-log-output .jp-OutputArea-child,
.dark-log-output .output_area,
.dark-log-output pre {
    background-color: #111 !important;
    color: white !important;
    border: none !important;
}
</style>
"""))

BASE_DATA_DIR = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data"
)
# One star per line: "Name" or "Name|RA_deg|Dec_deg" (explicit coordinates skip name lookup),
# same format as 01_Load_Cutout. Blank lines and lines starting with # are skipped.
TARGETS_FILE = BASE_DATA_DIR / "rcb_Bright_Targets.txt"

MAX_WORKERS = 6
SETUP_TIMEOUT_SECONDS = 900   # give up on a star if DASCH/name lookup gives no answer for this long
STALL_TIMEOUT_SECONDS = 300   # give up on a star whose downloads make ZERO progress for this long
                              # (a slow download that keeps finishing plates is never cut off)
MAX_REFCAT_SEP_ARCSEC = 5.0   # skip a star if DASCH's lightcurve source isn't at its position (None = off)

pause_button = widgets.Button(description="⏸ Pause", button_style="warning",
                               layout=widgets.Layout(width="120px", height="36px"))
progress_html = widgets.HTML(value="<i>Setting up...</i>")
top_bar = widgets.HBox([pause_button, progress_html],
                        layout=widgets.Layout(align_items="center", margin="0 0 12px 0"))
log_output = widgets.Output(layout=widgets.Layout(border="1px solid #444", padding="8px",
                                                    max_height="400px", overflow="auto"))
log_output.add_class("dark-log-output")

display(top_bar, log_output)

pause_event = threading.Event()
pause_button.on_click(lambda b: (pause_event.set(), setattr(pause_button, "description", "Pausing...")))


class StarSkipped(Exception):
    def __init__(self, short, detail):
        super().__init__(detail)
        self.short, self.detail = short, detail


def log(msg=""):
    with log_output:
        print(msg)

def log_block(text):
    text = (text or "").strip()
    if text:
        log("\n".join("  " + ln for ln in text.splitlines()))

def sanitize_folder_name(name):
    # same rule as 01_Load_Cutout, so folders line up with the ones it already made
    cleaned = re.sub(r'[<>:"/\\|?*]', '', name)
    cleaned = cleaned.strip()
    cleaned = re.sub(r'\s+', '_', cleaned)
    return cleaned

def format_eta(seconds):
    if seconds is None or seconds != seconds or seconds < 0:
        return "calculating..."
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:d}:{s:02d}"

def fmt_secs(s):
    return "n/a" if s is None else f"{format_eta(s)} ({s:,.2f}s)"

def pct(part, whole):
    return f"{part / whole * 100:5.1f}%" if whole else "   n/a"

def format_bytes(n):
    n = float(n)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"

def get_dir_size_bytes(path):
    total = 0
    if not path.exists():
        return 0
    for f in path.rglob("*"):
        if f.is_file():
            try:
                total += f.stat().st_size
            except OSError:
                pass
    return total

def render_progress(n_done, total, start_time, status_word="Downloading", color="#4CAF50", header=""):
    elapsed = time.monotonic() - start_time
    p = (n_done / total * 100) if total else 0
    rate = (n_done / elapsed) if elapsed > 0 and n_done > 0 else 0
    remaining_t = ((total - n_done) / rate) if rate > 0 else None
    filled = int(380 * p / 100)
    head = f'<div style="margin-bottom:4px;"><b>{html.escape(header)}</b></div>' if header else ""
    progress_html.value = f"""
    <div style="font-family: monospace; font-size: 14px; color: white;">
      {head}
      <div style="display:flex; align-items:center;">
        <div style="width:380px; height:18px; background:#333; border-radius:4px; overflow:hidden; margin-right:12px;">
          <div style="width:{filled}px; height:100%; background:{color};"></div>
        </div>
        <b>{p:5.1f}%</b>
      </div>
      <div style="margin-top:4px;">
        {status_word} &nbsp; <b>{n_done:,} / {total:,}</b> plates &nbsp;|&nbsp;
        <b>{rate:.2f}</b> plates/sec &nbsp;|&nbsp;
        Elapsed <b>{format_eta(elapsed)}</b> &nbsp;|&nbsp; ETA <b>{format_eta(remaining_t)}</b>
      </div>
    </div>
    """

def show_message(text, color="#4CAF50"):
    progress_html.value = (f'<div style="font-family: monospace; font-size: 14px; color: {color};">'
                           f'<b>{html.escape(text)}</b></div>')


def parse_object_list_file(path):
    entries, seen = [], set()
    for line in Path(path).read_text(encoding="utf-8-sig").splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue
        parts = [p.strip() for p in stripped.split("|")]
        name, ra_deg, dec_deg = parts[0], None, None
        if len(parts) == 3:
            try:
                ra_deg, dec_deg = float(parts[1]), float(parts[2])
            except ValueError:
                ra_deg = dec_deg = None
        elif len(parts) != 1:
            name = stripped
        key = sanitize_folder_name(name).lower()
        if key and key not in seen:
            seen.add(key)
            entries.append((name, ra_deg, dec_deg))
    return entries

def find_targets_file():
    if TARGETS_FILE.exists():
        return TARGETS_FILE
    root = BASE_DATA_DIR.parent
    for folder in (root / "notebooks", root, root / "project_Notes", root / "test_CNN" / "data"):
        if (folder / TARGETS_FILE.name).exists():
            return folder / TARGETS_FILE.name
    return None


SETUP_STEPS = ["check_previous_run", "open_session", "select_target", "select_refcat", "fetch_exposures",
               "exposures_to_pandas", "fetch_lightcurve", "apply_standard_rejections", "lightcurve_to_pandas",
               "refcat_position_check", "filter_accepted", "verify_mapping", "prepare_manifest"]
AFTER_STEPS = ["disk_scan"]
STEP_NOTES = {
    "check_previous_run": "reads plate_manifest.csv to see if an earlier run already finished this star",
    "open_session": "daschlab open_session()", "select_target": "name lookup / explicit coordinates",
    "select_refcat": "DASCH query for the APASS reference catalog", "fetch_exposures": "sess.exposures()",
    "exposures_to_pandas": "exposures table -> pandas", "fetch_lightcurve": "sess.lightcurve(0)",
    "apply_standard_rejections": "DASCH's standard automatic rejections", "lightcurve_to_pandas": "lightcurve -> pandas",
    "refcat_position_check": "is the lightcurve source actually at the star's position",
    "filter_accepted": "detected AND not rejected, exp_local_id -> exposure index",
    "verify_mapping": "series/platenum/mosnum/solnum check of every accepted row",
    "prepare_manifest": "load/create manifest, flag accepted plates, reconcile with files on disk",
    "download": "wall-clock time of the parallel cutout downloads", "disk_scan": "counting/measuring the cutouts folder",
}

class timed:
    def __init__(self, rec, step):
        self.rec, self.step = rec, step
    def __enter__(self):
        self.rec["current_step"] = self.step
        self.t0 = time.perf_counter()
    def __exit__(self, *exc):
        self.rec["timings"][self.step] = self.rec["timings"].get(self.step, 0.0) + time.perf_counter() - self.t0
        self.rec["current_step"] = None
        return False

def new_record(name, ra_deg, dec_deg):
    return {"name": name, "slug": sanitize_folder_name(name), "ra": ra_deg, "dec": dec_deg,
            "status": "not run", "note": "", "start_dt": None, "end_dt": None, "timings": {},
            "current_step": None, "plate_secs": [], "bytes": 0, "download_wall": 0.0,
            "run": {"downloaded": 0, "unavailable": 0, "error": 0}, "n_saves": 0, "save_seconds": 0.0,
            "errors": [], "refcat_sources": None, "funnel": {}, "counts": {}, "stats": {},
            "refcat_sep": None, "disk_files": None, "disk_bytes": None, "has_summary_file": False}

def run_with_timeout(fn, seconds, *args):
    box = {}
    def target():
        try:
            box["value"] = fn(*args)
        except BaseException as e:
            box["error"] = e
    t = threading.Thread(target=target, daemon=True)
    t.start()
    t.join(seconds)
    if t.is_alive():
        raise TimeoutError(f"no answer after {seconds}s")
    if "error" in box:
        raise box["error"]
    return box["value"]


def num_col(df, col):
    return pd.to_numeric(df[col], errors="coerce") if col in df.columns else pd.Series(dtype=float)

def date_span(df):
    try:
        parsed = pd.to_datetime(df["obs_date"], errors="coerce").dropna()
        return (str(parsed.min().date()), str(parsed.max().date())) if len(parsed) else ("unknown", "unknown")
    except Exception:
        return "unknown", "unknown"

def manifest_stats(df):
    out = {}
    for label, part in (("all", df), ("accepted", df[df["dasch_accepted"]])):
        ap, at = num_col(part, "lim_mag_apass"), num_col(part, "lim_mag_atlas")
        out[label] = {"rows": len(part), "n_apass": int(ap.notna().sum()),
                      "med_apass": f"{ap.median():.2f}" if ap.notna().any() else "--",
                      "n_atlas": int(at.notna().sum()),
                      "med_atlas": f"{at.median():.2f}" if at.notna().any() else "--",
                      "dates": date_span(part)}
    return out

def count_status(manifest_df, accepted_indices):
    s = manifest_df["status"].reindex(accepted_indices).fillna("not_attempted")
    vc = s.value_counts()
    counts = {"accepted": len(accepted_indices), "on_disk": int(vc.get("downloaded", 0)),
              "unavailable": int(vc.get("unavailable", 0)), "errors": int(vc.get("error", 0)),
              "not_attempted": int(vc.get("not_attempted", 0))}
    counts["extra_on_disk"] = int(((manifest_df["status"] == "downloaded") & (~manifest_df["dasch_accepted"])).sum())
    return counts

def check_finished(manifest_path):
    # (accepted, downloaded, unavailable) if an earlier run already finished this star, else None
    if not manifest_path.exists():
        return None
    try:
        m = pd.read_csv(manifest_path, usecols=["exposure_index", "status", "dasch_accepted"])
    except Exception:
        return None   # no dasch_accepted column (e.g. made by 01_Load_Cutout) or unreadable -> full setup
    acc = m[m["dasch_accepted"].astype(str).str.lower() == "true"]
    n_down = int((acc["status"] == "downloaded").sum())
    n_unav = int((acc["status"] == "unavailable").sum())
    if len(acc) and len(acc) - n_down - n_unav == 0:
        return len(acc), n_down, n_unav
    return None

def refcat_offset_arcsec(rec, lc_df):
    # arcsec between the star and the reference-catalog source its lightcurve belongs to
    # (None = couldn't check, which is never treated as a mismatch)
    try:
        import astropy.units as u
        from astropy.coordinates import SkyCoord
        ra = float(lc_df["catalog_ra"].dropna().iloc[0])
        dec = float(lc_df["catalog_dec"].dropna().iloc[0])
        cat = SkyCoord(ra * u.deg, dec * u.deg)
        if "pos.ra" in lc_df.columns:
            # if the catalog columns aren't in degrees they won't agree with the measured positions
            meas = SkyCoord(float(lc_df["pos.ra"].median()) * u.deg, float(lc_df["pos.dec"].median()) * u.deg)
            if cat.separation(meas).arcsec > 60:
                return None
        if rec["ra"] is not None and rec["dec"] is not None:
            target = SkyCoord(rec["ra"] * u.deg, rec["dec"] * u.deg)
        else:
            target = SkyCoord.from_name(rec["name"])
        return float(target.separation(cat).arcsec)
    except Exception:
        return None


def prepare_star(rec):
    name, slug = rec["name"], rec["slug"]
    session_dir = BASE_DATA_DIR / slug
    cutout_dir = session_dir / "cutouts"
    manifest_path = session_dir / "plate_manifest.csv"
    rec["existed_before"] = session_dir.exists()

    with timed(rec, "check_previous_run"):
        finished = check_finished(manifest_path)
    if finished is not None:
        n_acc, n_down, n_unav = finished
        rec["counts"] = {"accepted": n_acc, "on_disk": n_down, "unavailable": n_unav, "errors": 0,
                         "not_attempted": 0, "extra_on_disk": None}
        log(f"  Already finished in an earlier run: {n_down} on disk, {n_unav} unavailable, "
            f"of {n_acc} accepted -- nothing to do (no network used).")
        return {"state": "already_complete"}

    # daschlab prints its own progress lines -- run its calls inside log_output (like 01_Load_Cutout)
    # so they land in the log box. Output swallows exceptions raised inside it, so catch here and re-raise.
    err = None
    with log_output:
        try:
            with timed(rec, "open_session"):
                sess = open_session(str(session_dir))
            with timed(rec, "select_target"):
                if rec["ra"] is not None and rec["dec"] is not None:
                    print(f"Resolving via explicit coordinates (ra={rec['ra']:.5f}, dec={rec['dec']:.5f}) "
                          f"-- skipping name lookup.")
                    sess.select_target(ra_deg=rec["ra"], dec_deg=rec["dec"])
                else:
                    sess.select_target(name)
            with timed(rec, "select_refcat"):
                sess.select_refcat("apass")
                try:
                    rec["refcat_sources"] = len(sess.refcat())
                except Exception:
                    rec["refcat_sources"] = None
            with timed(rec, "fetch_exposures"):
                exposures = sess.exposures()
            with timed(rec, "exposures_to_pandas"):
                exposures_df = exposures.to_pandas()
            with timed(rec, "fetch_lightcurve"):
                lc = sess.lightcurve(0)
            with timed(rec, "apply_standard_rejections"):
                lc.apply_standard_rejections()
            with timed(rec, "lightcurve_to_pandas"):
                lc_df = lc.to_pandas()
        except BaseException as e:
            err = e
    if err is not None:
        raise err

    exposures_df.insert(0, "exposure_index", range(len(exposures_df)))
    n_total_lc = len(lc_df)
    n_detected = int(lc_df["magcal_magdep"].notna().sum())
    n_rejected = int((lc_df["reject"] != 0).sum())

    # lightcurve(0) is the reference-catalog source closest to the star. If the catalog has
    # nothing at the star's position, that is a neighbour, not the star.
    sep = None
    with timed(rec, "refcat_position_check"):
        if MAX_REFCAT_SEP_ARCSEC is not None:
            sep = refcat_offset_arcsec(rec, lc_df)
    rec["refcat_sep"] = sep
    if sep is None:
        log("  Couldn't check that the lightcurve source sits on the star (continuing).")
    else:
        log(f"  Lightcurve source is {sep:.1f} arcsec from the star.")
        if sep > MAX_REFCAT_SEP_ARCSEC:
            raise StarSkipped("neighbour source", f"nearest reference-catalog source is {sep:.1f} arcsec from the "
                              f"star (limit {MAX_REFCAT_SEP_ARCSEC}), so lightcurve(0) is a neighbour")

    with timed(rec, "filter_accepted"):
        accepted_df = lc_df[(lc_df["reject"] == 0) & (lc_df["magcal_magdep"].notna())].copy()
        n_accepted = len(accepted_df)
        # exp_local_id == -1 (or any negative) is DASCH's sentinel for "no exposure mapping
        # available for this row" -- these rows are simply excluded, quietly.
        accepted_df["exposure_index"] = (pd.to_numeric(accepted_df["exp_local_id"], errors="coerce")
                                           .fillna(-1).astype(int))
        unmapped_mask = accepted_df["exposure_index"] < 0
        n_unmapped = int(unmapped_mask.sum())
        accepted_df = accepted_df[~unmapped_mask].copy()
    log(f"  Exposures {len(exposures_df):,} | lightcurve rows {n_total_lc:,} | detected {n_detected:,} "
        f"| flagged by standard rejects {n_rejected:,} | accepted {n_accepted:,}")
    if n_unmapped:
        log(f"  {n_unmapped} accepted rows have no exposure mapping (exp_local_id == -1) "
            f"-- excluded from download, this is expected/normal.")

    with timed(rec, "verify_mapping"):
        exposures_indexed = exposures_df.set_index("exposure_index")
        genuine_mismatches = []
        for _, row in accepted_df.iterrows():
            idx = row["exposure_index"]
            if idx not in exposures_indexed.index:
                genuine_mismatches.append((idx, "index not found in exposures()"))
                continue
            exp_row = exposures_indexed.loc[idx]
            checks = [
                (str(row.get("series", "")).strip(), str(exp_row.get("series", "")).strip()),
                (int(row.get("platenum", -1)), int(exp_row.get("platenum", -2))),
                (int(row.get("mosnum", -1)), int(exp_row.get("mosnum", -2))),
                (int(row.get("solnum", -1)), int(exp_row.get("solnum", -2))),
            ]
            if any(a != b for a, b in checks):
                genuine_mismatches.append((idx, checks))
    rec["funnel"] = {"exposures": len(exposures_df), "lc_rows": n_total_lc, "detected": n_detected,
                     "rejected": n_rejected, "accepted": n_accepted, "unmapped": n_unmapped,
                     "mapped": len(accepted_df), "mismatches": len(genuine_mismatches)}
    if genuine_mismatches:
        raise StarSkipped("mapping mismatch", f"{len(genuine_mismatches)} GENUINE exposure-mapping mismatches "
                          f"(first: {genuine_mismatches[0]}) -- not downloading, don't trust this one")
    log(f"  Verified: all {len(accepted_df)} mapped rows match their exposure on "
        f"series/platenum/mosnum/solnum.")

    accepted_indices = sorted(accepted_df["exposure_index"].unique().tolist())
    session_dir.mkdir(parents=True, exist_ok=True)

    with timed(rec, "prepare_manifest"):
        if manifest_path.exists():
            manifest_df = pd.read_csv(manifest_path).dropna(subset=["exposure_index"])
            manifest_df["exposure_index"] = manifest_df["exposure_index"].astype(int)
            manifest_df = manifest_df.drop_duplicates("exposure_index", keep="last")
            # an older manifest can be shorter than the exposure list (DASCH added exposures)
            missing = exposures_df[~exposures_df["exposure_index"].isin(manifest_df["exposure_index"])]
            if len(missing):
                manifest_df = pd.concat([manifest_df, missing], ignore_index=True)
        else:
            manifest_df = exposures_df.copy()
        for col, default in (("status", "not_attempted"), ("filename", ""),
                             ("error_message", ""), ("last_updated", "")):
            if col not in manifest_df.columns:
                manifest_df[col] = default
            # an empty column reads back from csv as NaN floats -- make them real strings again
            manifest_df[col] = [default if pd.isna(v) else str(v) for v in manifest_df[col]]
        manifest_df["dasch_accepted"] = manifest_df["exposure_index"].astype(int).isin(accepted_indices)
        front_cols = [c for c in ["exposure_index", "dasch_accepted", "status", "filename",
                                  "lim_mag_apass", "lim_mag_atlas", "error_message", "last_updated"]
                      if c in manifest_df.columns]
        manifest_df = manifest_df[front_cols + [c for c in manifest_df.columns if c not in front_cols]]
        manifest_df = manifest_df.set_index("exposure_index", drop=False)

        existing_files = {f.name for f in cutout_dir.glob("*.fits")} if cutout_dir.exists() else set()
        if existing_files:
            on_disk_mask = pd.Series([bool(f) and Path(f).name in existing_files for f in manifest_df["filename"]],
                                     index=manifest_df.index)
            manifest_df.loc[(manifest_df["status"] != "downloaded") & on_disk_mask, "status"] = "downloaded"

        # "unavailable" = DASCH says it has no cutout image for that exposure, so it isn't re-asked
        status_of = manifest_df["status"].to_dict()
        remaining = [i for i in accepted_indices if status_of.get(i, "not_attempted") not in ("downloaded", "unavailable")]
        rec["stats"] = manifest_stats(manifest_df)
    already_have = len(accepted_indices) - len(remaining)
    n_extra = int(((manifest_df["status"] == "downloaded") & (~manifest_df["dasch_accepted"])).sum())
    log(f"  Of {len(accepted_indices)} DASCH-accepted, mapped plates: already handled {already_have}, "
        f"to download {len(remaining)}")
    if n_extra:
        log(f"  Note: {n_extra} cutouts already in this folder are NOT DASCH-accepted plates "
            f"(left untouched, but 02_A/03_A/03_B will process them too).")

    return {"state": "ready" if remaining else "nothing_to_download" if accepted_indices else "empty",
            "sess": sess, "manifest_df": manifest_df, "manifest_path": manifest_path,
            "accepted_indices": accepted_indices, "remaining": remaining}


def download_star(rec, info, header):
    sess, manifest_df = info["sess"], info["manifest_df"]
    remaining, manifest_path = info["remaining"], info["manifest_path"]
    total = len(remaining)
    n_done = 0
    start_time = time.monotonic()
    t_wall0 = time.perf_counter()
    state = {"last_save": 0.0, "warned": False}

    def fetch_one(i):
        t0 = time.perf_counter()
        try:
            result = sess.cutout(i)
            dt = time.perf_counter() - t0
            if result is None:
                return i, "unavailable", None, "no cutout available (likely unscanned)", dt, 0
            try:
                size = Path(result).stat().st_size
            except OSError:
                size = 0
            return i, "downloaded", str(result), None, dt, size
        except Exception as e:
            return i, "error", None, str(e), time.perf_counter() - t0, 0

    def save_manifest(force=False):
        if not force and time.monotonic() - state["last_save"] < 2.0:
            return
        state["last_save"] = time.monotonic()
        t0 = time.perf_counter()
        try:
            manifest_df.reset_index(drop=True).sort_values("exposure_index").to_csv(
                manifest_path, index=False
            )
        except Exception as e:
            if not state["warned"]:
                state["warned"] = True
                log(f"  Couldn't save the manifest right now ({e}). If it's open in Excel, close it "
                    f"-- will keep trying.")
        rec["n_saves"] += 1
        rec["save_seconds"] += time.perf_counter() - t0

    def record(fut):
        nonlocal n_done
        i, status, filename, err_msg, dt, size = fut.result()
        manifest_df.loc[i, "status"] = status
        manifest_df.loc[i, "filename"] = filename or ""
        manifest_df.loc[i, "error_message"] = err_msg or ""
        manifest_df.loc[i, "last_updated"] = datetime.now().isoformat(timespec="seconds")
        rec["run"][status] += 1
        rec["plate_secs"].append(dt)
        rec["bytes"] += size
        if status == "error":
            rec["errors"].append((i, err_msg))
        n_done += 1

    outcome = "completed"
    ex = ThreadPoolExecutor(max_workers=MAX_WORKERS)
    pending = {ex.submit(fetch_one, i) for i in remaining}
    last_progress = time.monotonic()
    render_progress(0, total, start_time, header=header)
    try:
        while pending:
            if pause_event.is_set():
                outcome = "paused"
                log("  Pause requested -- letting the downloads already in flight finish, then stopping.")
                ex.shutdown(wait=True, cancel_futures=True)
                for fut in pending:
                    if fut.done() and not fut.cancelled():
                        record(fut)
                break
            if time.monotonic() - last_progress > STALL_TIMEOUT_SECONDS:
                outcome = "stalled"
                log(f"  No plate finished for {STALL_TIMEOUT_SECONDS}s -- giving up on this star "
                    f"and moving on (run the cell again later to resume it).")
                ex.shutdown(wait=False, cancel_futures=True)
                break
            done, pending = wait(pending, timeout=0.2, return_when=FIRST_COMPLETED)
            for fut in done:
                record(fut)
            if done:
                last_progress = time.monotonic()
                render_progress(n_done, total, start_time, header=header)
                save_manifest()
    finally:
        ex.shutdown(wait=(outcome != "stalled"), cancel_futures=True)
        save_manifest(force=True)
        rec["download_wall"] = time.perf_counter() - t_wall0
        rec["timings"]["download"] = rec["download_wall"]

    render_progress(n_done, total, start_time, status_word=outcome.capitalize(),
                    color={"completed": "#4CAF50", "paused": "#FFA726", "stalled": "#E53935"}[outcome],
                    header=header)
    r = rec["run"]
    log(f"  {outcome.capitalize()} -- this run: downloaded {r['downloaded']}, unavailable {r['unavailable']}, "
        f"error {r['error']} in {rec['download_wall']:.1f}s")
    return outcome


def star_seconds(rec):
    if rec["start_dt"] and rec["end_dt"]:
        return (rec["end_dt"] - rec["start_dt"]).total_seconds()
    return None

def setup_seconds(rec):
    return sum(rec["timings"].get(s, 0.0) for s in SETUP_STEPS)

def latency_stats(secs):
    if not secs:
        return None
    a = np.asarray(secs, dtype=float)
    return {"n": len(a), "mean": a.mean(), "median": float(np.median(a)), "p90": float(np.percentile(a, 90)),
            "p95": float(np.percentile(a, 95)), "min": a.min(), "max": a.max(), "sum": a.sum()}

def build_star_text(rec):
    W = 78
    L = []
    def head(t):
        L.extend(["", "-" * W, t, "-" * W])
    L.extend(["=" * W, "DASCH-ACCEPTED PLATE CUTOUT DOWNLOAD SUMMARY", f"Object : {rec['name']}", "=" * W, ""])
    L.append(f"Run status                     : {rec['status']}")
    if rec["note"]:
        L.append(f"Note                           : {rec['note']}")
    L.append(f"Summary generated at           : {datetime.now().isoformat(timespec='seconds')}")

    head("TIMING")
    total = star_seconds(rec)
    L.append(f"Star run started               : {rec['start_dt'].isoformat(timespec='seconds') if rec['start_dt'] else 'n/a'}")
    L.append(f"Star run ended                 : {rec['end_dt'].isoformat(timespec='seconds') if rec['end_dt'] else 'n/a'}")
    L.append(f"Total time for this star       : {fmt_secs(total)}")
    L.append(f"  Setup (all DASCH queries)    : {fmt_secs(setup_seconds(rec))}   {pct(setup_seconds(rec), total or 0)} of total")
    L.append(f"  Downloading cutouts          : {fmt_secs(rec['download_wall'])}   {pct(rec['download_wall'], total or 0)} of total")
    L.append(f"    of which manifest saves    : {rec['save_seconds']:.2f}s over {rec['n_saves']} saves")
    L.append("")
    L.append(f"  {'Step':<28}{'Seconds':>12}{'% of star':>11}   What it does")
    for step in SETUP_STEPS + ["download"] + AFTER_STEPS:
        if step in rec["timings"]:
            s = rec["timings"][step]
            L.append(f"  {step:<28}{s:>12.3f}{pct(s, total or 0):>11}   {STEP_NOTES.get(step, '')}")
    if total:
        glue = max(total - setup_seconds(rec) - rec["timings"].get("download", 0.0) - rec["timings"].get("disk_scan", 0.0), 0.0)
        L.append(f"  {'(glue between steps)':<28}{glue:>12.3f}{pct(glue, total):>11}   time not inside any step above")
    if rec["current_step"]:
        L.append(f"  !! still inside step '{rec['current_step']}' when the run gave up on this star")

    stats = latency_stats(rec["plate_secs"])
    head("DOWNLOAD PERFORMANCE (this run)")
    if stats is None:
        L.append("No cutouts were requested in this run.")
    else:
        wall = rec["download_wall"]
        L.append(f"Cutouts requested              : {stats['n']:,}")
        L.append(f"Wall-clock download time       : {fmt_secs(wall)}")
        L.append(f"Throughput                     : {stats['n'] / wall:.3f} plates/sec" if wall > 0 else "Throughput                     : n/a")
        L.append(f"Time per plate (one request)   : mean {stats['mean']:.3f}s | median {stats['median']:.3f}s | "
                 f"p90 {stats['p90']:.3f}s | p95 {stats['p95']:.3f}s | min {stats['min']:.3f}s | max {stats['max']:.3f}s")
        L.append(f"Sum of per-request time        : {stats['sum']:.1f}s across {MAX_WORKERS} parallel workers")
        if wall > 0:
            L.append(f"Worker utilisation             : {stats['sum'] / (wall * MAX_WORKERS) * 100:.1f}% "
                     f"(100% = every worker busy the whole time)")
        L.append(f"Data downloaded (new files)    : {format_bytes(rec['bytes'])}")
        if wall > 0 and rec["bytes"]:
            L.append(f"Data rate                      : {rec['bytes'] / wall / 1024 / 1024:.2f} MB/s")
        dl = rec["run"]["downloaded"]
        if dl:
            L.append(f"Average file size              : {format_bytes(rec['bytes'] / dl)}")

    head("DASCH LIGHTCURVE FUNNEL (why these plates)")
    f = rec["funnel"]
    if f:
        if rec["refcat_sources"] is not None:
            L.append(f"APASS reference sources in field : {rec['refcat_sources']:,}")
        L.append(f"Exposures DASCH lists for target : {f['exposures']:,}")
        L.append(f"Lightcurve rows                  : {f['lc_rows']:,}")
        L.append(f"Detected (magcal_magdep present) : {f['detected']:,}")
        L.append(f"Flagged by standard rejections   : {f['rejected']:,}")
        L.append(f"ACCEPTED (detected, not rejected): {f['accepted']:,}")
        L.append(f"  no exposure mapping (-1)       : {f['unmapped']:,}   (excluded, expected/normal)")
        L.append(f"  mapped to a real exposure      : {f['mapped']:,}   <- the plates this notebook downloads")
        L.append(f"  mapping mismatches             : {f['mismatches']:,}")
    else:
        L.append("Not computed this run (star was finished earlier, or setup did not get that far).")
    if rec["refcat_sep"] is not None:
        L.append(f"Lightcurve source distance from star : {rec['refcat_sep']:.2f} arcsec (limit {MAX_REFCAT_SEP_ARCSEC})")

    head("THIS RUN")
    r = rec["run"]
    L.append(f"Downloaded this run            : {r['downloaded']:,}")
    L.append(f"Unavailable this run           : {r['unavailable']:,}")
    L.append(f"Errored this run               : {r['error']:,}")

    head("ALL-TIME TOTALS (DASCH-accepted plates, across every run)")
    c = rec["counts"]
    if c:
        n = c["accepted"]
        L.append(f"DASCH-accepted plates          : {n:,}")
        for label, key in (("On disk (downloaded)", "on_disk"), ("Unavailable (no cutout at DASCH)", "unavailable"),
                           ("Errored", "errors"), ("Not attempted yet", "not_attempted")):
            L.append(f"{label:<31}: {c[key]:,}  ({pct(c[key], n).strip()})")
        if c.get("extra_on_disk") is not None:
            L.append(f"Extra cutouts on disk that are NOT DASCH-accepted : {c['extra_on_disk']:,}")
    if rec["disk_files"] is not None:
        L.append(f"FITS files currently in cutouts/ : {rec['disk_files']:,}   ({format_bytes(rec['disk_bytes'])})")

    if rec["stats"]:
        head("PLATE LIMITS (limiting magnitude) AND OBSERVATION DATES")
        for label, title in (("accepted", "DASCH-accepted plates"), ("all", "every exposure DASCH lists")):
            s = rec["stats"][label]
            L.append(f"{title} ({s['rows']:,} rows):")
            L.append(f"  Plates with lim_mag_apass / median : {s['n_apass']:,} / {s['med_apass']}")
            L.append(f"  Plates with lim_mag_atlas / median : {s['n_atlas']:,} / {s['med_atlas']}")
            L.append(f"  Observation dates                  : {s['dates'][0]}  to  {s['dates'][1]}")

    head("SETTINGS FOR THIS RUN")
    L.append(f"Download workers               : {MAX_WORKERS}")
    L.append(f"Setup timeout                  : {SETUP_TIMEOUT_SECONDS}s")
    L.append(f"Stall timeout (zero progress)  : {STALL_TIMEOUT_SECONDS}s")
    L.append(f"Lightcurve-source distance limit : {MAX_REFCAT_SEP_ARCSEC} arcsec")

    if rec["errors"]:
        head(f"ERRORED PLATES (first {min(len(rec['errors']), 25)} of {len(rec['errors'])} this run)")
        for i, msg in rec["errors"][:25]:
            L.append(f"  exposure {i}: {msg}")

    head("FILE LOCATIONS")
    L.append(f"Session directory              : {BASE_DATA_DIR / rec['slug']}")
    L.append(f"Cutouts (the .fits files)      : {BASE_DATA_DIR / rec['slug'] / 'cutouts'}")
    L.append(f"Manifest (one row per exposure): {BASE_DATA_DIR / rec['slug'] / 'plate_manifest.csv'}   (dasch_accepted column marks the plates DASCH uses)")
    L.append(f"This text summary              : {BASE_DATA_DIR / rec['slug'] / (rec['slug'] + '_Cutout_Download_Summary.txt')}")
    L.extend(["", "=" * W])
    return "\n".join(L)


def quick_table(recs):
    hdr = (f"{'Object':<20}{'Status':<26}{'Accepted':>9}{'OnDisk':>8}{'Unavail':>8}{'Errors':>7}"
           f"{'Setup':>9}{'Download':>10}{'Total':>9}{'Storage':>11}")
    L = [hdr]
    for r in recs:
        c = r["counts"]
        L.append(f"{r['name'][:19]:<20}{r['status'][:25]:<26}"
                 f"{c.get('accepted', 0):>9,}{c.get('on_disk', 0):>8,}{c.get('unavailable', 0):>8,}{c.get('errors', 0):>7,}"
                 f"{format_eta(setup_seconds(r)) if r['timings'] else '-':>9}"
                 f"{format_eta(r['download_wall']) if r['download_wall'] else '-':>10}"
                 f"{format_eta(star_seconds(r)) if star_seconds(r) is not None else '-':>9}"
                 f"{format_bytes(r['disk_bytes']) if r['disk_bytes'] is not None else '-':>11}")
    return L

def env_info():
    def ver(pkg):
        try:
            return metadata.version(pkg)
        except Exception:
            return "not found"
    return [f"Python                : {sys.version.split()[0]} ({platform.platform()})",
            f"daschlab              : {ver('daschlab')}",
            f"astropy / numpy / pandas : {ver('astropy')} / {ver('numpy')} / {ver('pandas')}"]

def build_batch_text(recs, batch_start, batch_end, targets_path):
    W = 78
    end = batch_end or datetime.now()
    wall = (end - batch_start).total_seconds()
    L = ["=" * W, "BATCH DASCH-ACCEPTED PLATE CUTOUT EXTRACTION SUMMARY (verbose)", "=" * W, ""]
    L.append(f"Batch start time         : {batch_start.isoformat(timespec='seconds')}")
    L.append(f"Batch end time           : {end.isoformat(timespec='seconds') if batch_end else 'still running / interrupted'}")
    L.append(f"Batch total duration     : {fmt_secs(wall)}")
    L.append(f"Objects requested        : {len(recs)}   (from {targets_path})")
    L.append(f"Data folder              : {BASE_DATA_DIR}")
    L.append(f"Download workers         : {MAX_WORKERS}")
    L.append(f"Setup timeout            : {SETUP_TIMEOUT_SECONDS}s per star")
    L.append(f"Stall timeout            : {STALL_TIMEOUT_SECONDS}s of zero progress (NOT a cap on total time)")
    L.append(f"Lightcurve-source limit  : {MAX_REFCAT_SEP_ARCSEC} arcsec")
    L.extend(env_info())

    L.extend(["", "-" * W, "QUICK SUMMARY (one row per object)", "-" * W])
    L.extend(quick_table(recs))
    tot = {k: sum(r["counts"].get(k, 0) or 0 for r in recs) for k in ("accepted", "on_disk", "unavailable", "errors")}
    L.append("-" * W)
    L.append(f"{'TOTAL':<46}{tot['accepted']:>9,}{tot['on_disk']:>8,}{tot['unavailable']:>8,}{tot['errors']:>7,}")

    L.extend(["", "-" * W, "WHERE THE TIME WENT (all objects added together)", "-" * W])
    L.append(f"  {'Step':<28}{'Seconds':>12}{'% of batch':>12}")
    accounted = 0.0
    for step in SETUP_STEPS + ["download"] + AFTER_STEPS:
        s = sum(r["timings"].get(step, 0.0) for r in recs)
        if s > 0:
            accounted += s
            L.append(f"  {step:<28}{s:>12.2f}{pct(s, wall):>12}")
    write_s = sum(r["timings"].get("write_star_summary", 0.0) for r in recs)
    L.append(f"  {'write_star_summary':<28}{write_s:>12.2f}{pct(write_s, wall):>12}")
    other = max(wall - accounted - write_s, 0.0)
    L.append(f"  {'other (UI, report writing)':<28}{other:>12.2f}{pct(other, wall):>12}")
    L.append(f"  {'TOTAL WALL-CLOCK':<28}{wall:>12.2f}{'100.0%':>12}")

    all_secs = [s for r in recs for s in r["plate_secs"]]
    stats = latency_stats(all_secs)
    L.extend(["", "-" * W, "DOWNLOAD PERFORMANCE (all objects, this run)", "-" * W])
    if stats is None:
        L.append("No cutouts were requested in this run.")
    else:
        dl_wall = sum(r["download_wall"] for r in recs)
        total_bytes = sum(r["bytes"] for r in recs)
        L.append(f"Cutouts requested        : {stats['n']:,}")
        L.append(f"Wall-clock downloading   : {fmt_secs(dl_wall)}")
        L.append(f"Overall throughput       : {stats['n'] / dl_wall:.3f} plates/sec" if dl_wall > 0 else "Overall throughput       : n/a")
        L.append(f"Time per plate           : mean {stats['mean']:.3f}s | median {stats['median']:.3f}s | "
                 f"p90 {stats['p90']:.3f}s | p95 {stats['p95']:.3f}s | min {stats['min']:.3f}s | max {stats['max']:.3f}s")
        L.append(f"Data downloaded          : {format_bytes(total_bytes)}")
        if dl_wall > 0 and total_bytes:
            L.append(f"Data rate                : {total_bytes / dl_wall / 1024 / 1024:.2f} MB/s")
        L.append(f"Average time per plate (wall-clock, all objects) : {dl_wall / stats['n']:.3f}s")

    L.extend(["", "-" * W, "TIMELINE", "-" * W])
    for n, r in enumerate(recs, 1):
        s = r["start_dt"].strftime("%H:%M:%S") if r["start_dt"] else "--:--:--"
        e = r["end_dt"].strftime("%H:%M:%S") if r["end_dt"] else "--:--:--"
        L.append(f"  {n:>2}. {r['name']:<24}{s} -> {e}   {fmt_secs(star_seconds(r))}   [{r['status']}]")

    problems = [r for r in recs if r["status"].startswith(("skipped", "failed", "stalled", "not started"))]
    if problems:
        L.extend(["", "-" * W, "SKIPPED / FAILED / STALLED / NOT STARTED", "-" * W])
        for r in problems:
            L.append(f"  {r['name']}: {r['status']}")
            if r["note"]:
                L.append(f"      {r['note']}")
            if r["current_step"]:
                L.append(f"      was inside step '{r['current_step']}'")

    L.extend(["", "=" * W, "PER-OBJECT DETAIL", "=" * W])
    for r in recs:
        if r["timings"] or r["counts"]:
            L.extend(["", build_star_text(r)])
        else:
            L.extend(["", f"OBJECT: {r['name']}  -- {r['status']}"])
    L.extend(["", "=" * W, "END OF REPORT", "=" * W])
    return "\n".join(L)


def run_batch():
    finish_word, finish_color = "Done", "#4CAF50"
    recs, batch_start, batch_end, targets_path = [], datetime.now(), None, None

    def write_batch_report(final=False):
        if not recs:
            return
        try:
            path = BASE_DATA_DIR / f"download_Summary_{batch_start.strftime('%Y%m%d_%H%M%S')}.txt"
            path.write_text(build_batch_text(recs, batch_start, batch_end if final else None, targets_path),
                            encoding="utf-8")
            return path
        except Exception as e:
            log(f"Couldn't write the batch report right now ({e}).")

    try:
        BASE_DATA_DIR.mkdir(parents=True, exist_ok=True)
        targets_path = find_targets_file()
        if targets_path is None:
            log(f"Targets file not found: {TARGETS_FILE}\n"
                f"Put your list there, or change TARGETS_FILE at the top of this cell.")
            finish_word, finish_color = "Stopped", "#E53935"
            return
        entries = parse_object_list_file(targets_path)
        if not entries:
            log(f"No star names found in {targets_path.name}.")
            finish_word, finish_color = "Stopped", "#E53935"
            return
        recs.extend(new_record(*e) for e in entries)
        log(f"{len(recs)} stars from {targets_path}: {', '.join(r['name'] for r in recs)}")
        log("(DASCH's own warning about apply_standard_rejections(): this is under development, "
            "treat with appropriate caution)")

        for n, rec in enumerate(recs, 1):
            name = rec["name"]
            if pause_event.is_set():
                finish_word, finish_color = "Paused", "#FFA726"
                for rest in recs[n - 1:]:
                    rest["status"] = "not started (paused)"
                break

            header = f"Star {n}/{len(recs)}: {name}"
            log(f"\n[{n}/{len(recs)}] {name}")
            render_progress(0, 1, time.monotonic(), status_word="Setting up", color="#42A5F5", header=header)
            rec["start_dt"] = datetime.now()
            outcome, info, timed_out = None, None, False

            try:
                info = run_with_timeout(prepare_star, SETUP_TIMEOUT_SECONDS, rec)
                if info["state"] == "already_complete":
                    rec["status"] = "already complete (skipped)"
                elif info["state"] == "empty":
                    rec["status"] = "no accepted plates"
                else:
                    if info["state"] == "ready":
                        outcome = download_star(rec, info, header)
                    with timed(rec, "disk_scan"):
                        rec["counts"] = count_status(info["manifest_df"], info["accepted_indices"])
                        cdir = BASE_DATA_DIR / rec["slug"] / "cutouts"
                        rec["disk_files"] = len(list(cdir.glob("*.fits"))) if cdir.exists() else 0
                        rec["disk_bytes"] = get_dir_size_bytes(cdir)
                    c = rec["counts"]
                    if outcome == "paused":
                        rec["status"] = "partial (paused)"
                    elif outcome == "stalled":
                        rec["status"] = "stalled"
                        rec["note"] = (f"no plate finished for {STALL_TIMEOUT_SECONDS}s -- downloads abandoned for now; "
                                       f"run the cell again to resume this star")
                    else:
                        rec["status"] = ("complete" if c["errors"] == 0 and c["not_attempted"] == 0
                                         else "complete (some errors)")
            except StarSkipped as e:
                rec["status"], rec["note"] = f"skipped: {e.short}", e.detail
                log(f"  Skipped -- {e.detail}")
            except TimeoutError:
                timed_out = True
                rec["status"] = "failed: setup timed out"
                rec["note"] = (f"no answer after {SETUP_TIMEOUT_SECONDS}s while in step "
                               f"'{rec['current_step']}' -- run the cell again to retry this star")
                log(f"  Failed -- {rec['note']}")
            except Exception as e:
                globals()["last_error_traceback"] = traceback.format_exc()
                rec["status"] = f"failed: {type(e).__name__}"
                rec["note"] = f"{e}" + (f"   (during step '{rec['current_step']}')" if rec["current_step"] else "")
                log(f"  Failed -- {type(e).__name__}: {e}")

            rec["end_dt"] = datetime.now()
            sdir = BASE_DATA_DIR / rec["slug"]
            if not timed_out and rec["status"] != "already complete (skipped)":
                if rec["status"].startswith(("skipped", "failed")):
                    # don't leave an empty session folder behind for a star that never got going
                    if (not rec.get("existed_before") and sdir.exists()
                            and not (sdir / "plate_manifest.csv").exists() and not list(sdir.rglob("*.fits"))):
                        shutil.rmtree(sdir, ignore_errors=True)
                else:
                    if not rec["counts"] and (sdir / "cutouts").is_dir() and not any((sdir / "cutouts").iterdir()):
                        (sdir / "cutouts").rmdir()   # no plates to keep: 02_A/03_A/03_B would list it as a 0-plate star
                    t0 = time.perf_counter()
                    try:
                        (sdir / f"{rec['slug']}_Cutout_Download_Summary.txt").write_text(
                            build_star_text(rec), encoding="utf-8")
                    except Exception as e:
                        log(f"  Couldn't write the per-star summary ({e}).")
                    rec["timings"]["write_star_summary"] = time.perf_counter() - t0

            log(f"  [{rec['status']}]  setup {format_eta(setup_seconds(rec))} | "
                f"download {format_eta(rec['download_wall'])} | total {format_eta(star_seconds(rec))}")
            write_batch_report()
            if outcome == "paused":
                finish_word, finish_color = "Paused", "#FFA726"
                for rest in recs[n:]:
                    rest["status"] = "not started (paused)"
                break
    except Exception:
        globals()["last_error_traceback"] = traceback.format_exc()
        log("Batch stopped by an unexpected error:\n" + traceback.format_exc())
        finish_word, finish_color = "Error", "#E53935"
    finally:
        batch_end = datetime.now()
        report_path = write_batch_report(final=True)
        if recs:
            log("\nSummary:")
            log("\n".join(quick_table(recs)))
            log(f"\nTotal wall-clock: {fmt_secs((batch_end - batch_start).total_seconds())}")
            if report_path:
                log(f"Full timing report: {report_path}")
        pause_button.description = finish_word
        pause_button.disabled = True
        show_message(f"{finish_word} -- {sum(1 for r in recs if r['start_dt'])} stars processed", finish_color)


_batch_thread = threading.Thread(target=run_batch, daemon=True)
_batch_thread.start()


Output(layout=Layout(border_bottom='1px solid #444', border_left='1px solid #444', border_right='1px solid #44…

- Querying API ...
- Querying API ...
- Querying API ...
- Querying API ...
- Querying API ...
- Querying API ...
- Fetched 1399680 bytes in 6 seconds
- Fetched 1399680 bytes in 6 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V854_Cen\cutouts\00603_am05354m0s0.fits`
- Querying API ...
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V854_Cen\cutouts\00557_am04318m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 6 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V854_Cen\cutouts\00478_am02794m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 6 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V854_Cen\cutouts\00473_am02748m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 6 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V854_Cen\cutouts\00416_am01914m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in

    (218.70584353, -39.55533369)> appears to have failed: {'message': 'Internal server error'}


- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V854_Cen\cutouts\04890_bi03695m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V854_Cen\cutouts\04891_bi03696m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V854_Cen\cutouts\04904_bi03733m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V854_Cen\cutouts\04906_rb15169m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V854_Cen\cutouts\04907_bi03742m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V854_Cen\cutouts\04914_bi037

    (218.70584353, -39.55533369)> appears to have failed: {'message': 'Endpoint request timed out'}


...

- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\03814_mf15734m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\03822_mf15773m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\03823_mf15774m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\03824_mf15775m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\03826_mf15776m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\03828

    (260.81064512, -22.86842071)> appears to have failed: {'message': 'Internal server error'}


- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\06644_rb13705m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\06651_rb13752m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 12 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\06618_a24869m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\06653_rb13767m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 9 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\06637_a24877m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\06701_

    (260.81064512, -22.86842071)> appears to have failed: {'message': 'Endpoint request timed out'}


- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\06918_bi02661m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\06941_b71683m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\06924_rb14393m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\06984_rb14450m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\07066_ax04473m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\V2552_Oph\cutouts\07081_